# 02.01 PyPTO 算子开发基础知识

本章建立 PyPTO 算子开发的基础认知。PyPTO 算子可以先理解为“用 Python 语法描述一段 Tensor 计算，再由框架把这段计算交给设备执行”的程序单元。后续初级教程会继续展开逐元素算子、矩阵乘法、规约和复杂算子封装；本章先从最小闭环开始，说明一个算子如何定义、如何准备输入输出、如何运行、如何验证。

本章围绕四个核心问题展开：

1. 一个最小 PyPTO 算子由哪些部分组成。
2. Host 侧代码和 Kernel 侧代码如何分工。
3. PyPTO API 如何描述参数、计算、Tile 和运行模式。
4. Tensor 表达如何形成计算图，并进一步进入 MPMD 执行模型。


## 1. 本章知识路径

PyPTO 算子开发可以先按下面这条主线理解：

```text
计算目标
  -> Kernel 定义
  -> 输入输出描述
  -> Tile Shape 设置
  -> Host 侧数据准备
  -> Kernel 调用
  -> PyTorch 参考验证
```

Hello World 加法算子的数学目标是 `out = x + y`。这个目标很小，但它覆盖了 PyPTO 开发的完整骨架：`@pypto.frontend.jit` 标记 kernel，`pypto.Tensor` 描述参数，`pypto.set_vec_tile_shapes` 描述执行组织，Host 侧用 PyTorch 创建真实输入输出，最后用 PyTorch 参考结果验证。

| 步骤 | 含义 | Hello World 中的对应内容 |
| --- | --- | --- |
| 明确计算目标 | 写清楚 Tensor 之间的数学关系 | `out = x + y` |
| 定义 kernel | 标记一段需要被 PyPTO 处理的函数 | `@pypto.frontend.jit(...)` |
| 描述输入输出 | 说明参数的 dtype 和 shape 特征 | `pypto.Tensor([...], pypto.DT_FP32)` |
| 设置 Tile Shape | 描述设备侧分块组织方式 | `pypto.set_vec_tile_shapes(1, 4, 1, 64)` |
| 准备数据 | 创建真实输入和输出 Tensor | `torch.rand`、`torch.empty` |
| 调用并验证 | 执行 kernel 并比较参考结果 | `assert_allclose` |

后续更复杂的算子仍然沿着这条主线展开，只是计算表达式、shape 关系、dtype 选择和 Tile 设置会更加丰富。


## 2. 本章 Notebook 关系

本章各节承担不同角色：

| Notebook | 作用 |
| --- | --- |
| `02.01_chapter_intro.ipynb` | 建立章节入口、开发主线和最小运行时检查 |
| `02.02_run_hello_world.ipynb` | 完整展开 Hello World 加法算子 |
| `02.03_programming_paradigm_mpmd.ipynb` | 解释 Host/Kernel 划分、PyPTO API、计算图和 MPMD |
| `02.04_api_and_compute_graph.ipynb` | 汇总常用 API 分类，并把计算图层次和开发闭环串联起来 |

这个顺序体现了从实践到概念、再回到整体框架的学习路径：先看到一个完整算子，再解释它背后的编程范式，最后建立 API 和计算图的索引。


## 3. 最小运行时检查

本章入口只需要确认 Python 环境能够导入 `torch` 和 `pypto`，并识别是否存在可选的 `torch_npu` 扩展。这里不查找示例源码位置，也不绑定固定目录；Notebook 中的示例代码应保持与机器路径无关。


In [1]:
import os
import torch
import pypto

try:
    import torch_npu
except ImportError:
    torch_npu = None


def describe_runtime():
    print("python runtime ready")
    print("torch:", torch.__version__)
    print("pypto:", pypto.__file__)
    print("torch_npu:", "available" if torch_npu is not None else "not available")
    print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))

    if torch_npu is None:
        print("default device kind: cpu / sim")
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    torch.npu.set_device(device_id)
    device = f"npu:{device_id}"
    print("default device:", device)
    return device


device = describe_runtime()


/usr/local/lib/python3.12/dist-packages/torch_npu/__init__.py:324: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


python runtime ready
torch: 2.7.1+cpu
pypto: /usr/local/lib/python3.12/dist-packages/pypto/__init__.py
torch_npu: available
TILE_FWK_DEVICE_ID: <未设置，默认 0>
default device: npu:0


这段检查只回答三个问题：`torch` 是否可用，`pypto` 是否可用，当前环境是否带有 `torch_npu`。如果 `torch_npu` 不存在，后续示例可以使用 SIM 模式理解流程；如果 `torch_npu` 存在，设备编号默认来自 `TILE_FWK_DEVICE_ID`，未设置时使用进程内的 `0`。

需要注意的是，`TILE_FWK_DEVICE_ID=0` 表示当前进程可见设备中的第 0 张，并不一定等同于物理机器上的第 0 张卡。可见设备范围通常由外部环境决定。


## 4. 本节小结

本节建立了 PyPTO 算子开发的主线：先定义计算目标，再用 JIT kernel 描述 Tensor 运算，随后由 Host 侧准备数据、调用 kernel 并完成验证。下一节的 Hello World 加法算子会把这条主线落到一段完整代码中；再往后，编程范式、MPMD、API 分类和计算图层次会解释这段代码为什么这样组织。
